# 02 — RFM Customer Segmentation
Segmenting customers by Recency, Frequency, Monetary value.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from src.rfm import load_data, compute_rfm, segment_summary, SEGMENT_COLORS
import warnings; warnings.filterwarnings('ignore')

engine = create_engine('sqlite:///../data/ecommerce.db')
sns.set_theme(style='whitegrid')
print("Ready")

## 1. Load Transactions & Compute RFM

In [ ]:
df  = load_data(engine)
rfm = compute_rfm(df)
print(f"Customers scored: {len(rfm):,}")
rfm[['customer_id','recency','frequency','monetary','R','F','M','rfm_score','segment']].head(10)

## 2. Segment Distribution

In [ ]:
summary = segment_summary(rfm)
print(summary[['segment','customers','customer_pct','total_revenue','revenue_pct']].to_string(index=False))

# Revenue share — headline finding
top_rev = summary[summary['segment'].isin(['Champions','Loyal Customers'])]['total_revenue'].sum()
total   = summary['total_revenue'].sum()
print(f"\n★ Champions + Loyal → {top_rev/total*100:.1f}% of total revenue")

## 3. RFM Score Distributions

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5))
for i, (col, title, color) in enumerate([
        ('R','Recency Score','#e74c3c'),
        ('F','Frequency Score','#2980b9'),
        ('M','Monetary Score','#2ecc71')]):
    counts = rfm[col].value_counts().sort_index()
    axes[i].bar(counts.index, counts.values, color=color, alpha=0.85)
    axes[i].set_title(title); axes[i].set_xlabel('Score (1=worst, 5=best)')
    axes[i].set_ylabel('Customers')
plt.suptitle('RFM Score Distributions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Segment Revenue Treemap

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14,8))
ax.set_xlim(0,100); ax.set_ylim(0,100); ax.axis('off')
ax.set_title('Revenue Treemap by Customer Segment', fontsize=14, fontweight='bold')

total_rev = summary['total_revenue'].sum()
x, y, row_h = 0, 100, 0
for _, row in summary.iterrows():
    w = row['revenue_pct']
    h = 100 * row['customers'] / summary['customers'].sum() * 2
    h = max(h, 8)
    if x + w > 100:
        x=0; y -= row_h; row_h=0
    color = SEGMENT_COLORS.get(row['segment'],'#888')
    rect = plt.Rectangle((x,y-h), w, h, facecolor=color, edgecolor='white', lw=2)
    ax.add_patch(rect)
    if w > 5:
        ax.text(x+w/2, y-h/2, f"{row['segment']}\n{row['customers']:,} cust\n{row['revenue_pct']:.1f}%",
                ha='center', va='center', fontsize=8, fontweight='bold', color='white',
                wrap=True)
    x += w; row_h = max(row_h, h)
plt.tight_layout(); plt.show()

## 5. RFM Heatmap (R × F customer density)

In [ ]:
pivot = rfm.groupby(['R','F']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            cbar_kws={'label':'Customer Count'})
ax.set_title('Customer Density: Recency Score × Frequency Score')
ax.set_xlabel('Frequency Score (1=low, 5=high)')
ax.set_ylabel('Recency Score (1=old, 5=recent)')
plt.tight_layout(); plt.show()

## 6. Segment Profiles

In [ ]:
profile = rfm.groupby('segment').agg(
    n=('customer_id','count'),
    avg_recency=('recency','mean'),
    avg_frequency=('frequency','mean'),
    avg_spend=('monetary','mean'),
    median_spend=('monetary','median')
).round(1)
profile['avg_recency'] = profile['avg_recency'].astype(int)
print(profile.sort_values('avg_spend',ascending=False).to_string())

## 7. Export & Resume Impact

In [ ]:
rfm.to_csv('../outputs/rfm_segments.csv', index=False)
summary.to_csv('../outputs/rfm_segment_summary.csv', index=False)
print("Exported rfm_segments.csv and rfm_segment_summary.csv")

# Print the exact resume bullet
top2_cust = summary[summary['segment'].isin(['Champions','Loyal Customers'])]['customers'].sum()
top2_rev  = summary[summary['segment'].isin(['Champions','Loyal Customers'])]['total_revenue'].sum()
total_rev  = summary['total_revenue'].sum()
total_cust = summary['customers'].sum()
print(f"\n★ RESUME BULLET:")
print(f"Analyzed {len(df):,} transactions across {total_cust:,} customers;")
print(f"RFM segmentation identified {top2_cust:,} high-value customers")
print(f"contributing {top2_rev/total_rev*100:.0f}% of total revenue.")